<a href="https://colab.research.google.com/github/zoranmrsa/FirstRepository/blob/main/AlphaFold2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Instalacija ColabFold
import os, sys

# Instaliraj ColabFold
os.system("pip install -q --no-warn-conflicts \
  'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
os.system("pip install -q --upgrade tensorflow")
os.system("pip install -q silence_tensorflow")
os.system("pip install -q --upgrade 'jax[cuda12_pip]' \
  -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")

print("✅ Instalacija gotova")

In [ ]:
import requests, os, time

jobname = "GBM_trimmed"
os.makedirs(jobname, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "colabfold-helper/1.0"})

def get_json(uniprot_id, retries=3, backoff=1.0):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt + 1 == retries:
                raise
            time.sleep(backoff * (attempt + 1))

def get_fasta_seq(uniprot_id):
    url = f"https://www.uniprot.org/uniprot/{uniprot_id}.fasta"
    r = session.get(url, timeout=20)
    r.raise_for_status()
    lines = r.text.splitlines()
    return "".join(lines[1:])

def extract_ecd_from_json(j):
    full_seq = j['sequence']['value']
    signal_end_1 = None
    tm_first_start_1 = None
    for feat in j.get('features', []):
        t = feat.get('type', '').lower()
        loc = feat.get('location', {})
        start_val = loc['start'].get('value') if isinstance(loc.get('start'), dict) else None
        end_val   = loc['end'].get('value')   if isinstance(loc.get('end'),   dict) else None
        if isinstance(start_val, str): start_val = int(start_val)
        if isinstance(end_val,   str): end_val   = int(end_val)
        if t == 'signal' and end_val is not None:
            signal_end_1 = end_val
        if t == 'transmembrane' and start_val is not None:
            if tm_first_start_1 is None or start_val < tm_first_start_1:
                tm_first_start_1 = start_val
    ecd_start_idx = signal_end_1 if signal_end_1 is not None else 0
    if tm_first_start_1 is not None:
        ecd_end_idx = tm_first_start_1 - 2
        if ecd_end_idx < ecd_start_idx:
            ecd_end_idx = len(full_seq) - 1
    else:
        ecd_end_idx = len(full_seq) - 1
    ecd_seq = full_seq[ecd_start_idx: ecd_end_idx + 1]
    return full_seq, ecd_seq, ecd_start_idx, ecd_end_idx, signal_end_1, tm_first_start_1

# UniProt ID-ovi
proteins = {
    "CLIC1": "P45736",
    "OSMR":  "Q99650",
    "EGFR":  "P00533",
}

fetched = {}
print("Fetching sequences from UniProt...")
for name, uid in proteins.items():
    j = get_json(uid)
    full, ecd, ecd_si, ecd_ei, sig_end, tm_start = extract_ecd_from_json(j)
    fetched[name] = {
        "uniprot": uid, "full_seq": full, "ecd_seq": ecd,
        "ecd_start_idx": ecd_si, "ecd_end_idx": ecd_ei,
        "signal_end_1based": sig_end, "tm_start_1based": tm_start,
    }
    print(f"  {name} ({uid}): full={len(full)}, ECD={len(ecd)}, TM_start={tm_start}")

# EGFRvIII: delecija AA 6-273 (0-based: 5..272)
del_start_0, del_end_0 = 5, 272
egfr_full = fetched["EGFR"]["full_seq"]
egfrviii_full = egfr_full[:del_start_0] + egfr_full[del_end_0 + 1:]
fetched["EGFR"]["viii_full_seq"] = egfrviii_full

# EGFRvIII ECD
egfr_info = fetched["EGFR"]
orig_ecd  = egfr_info["ecd_seq"]
si, ei    = egfr_info["ecd_start_idx"], egfr_info["ecd_end_idx"]
ov_s = max(si, del_start_0)
ov_e = min(ei, del_end_0)
if ov_s <= ov_e:
    rs, re = ov_s - si, ov_e - si
    egfrviii_ecd = orig_ecd[:rs] + orig_ecd[re + 1:]
else:
    egfrviii_ecd = orig_ecd
fetched["EGFR"]["viii_ecd_seq"] = egfrviii_ecd
print(f"\nEGFRvIII: full={len(egfrviii_full)}, ECD={len(egfrviii_ecd)}")

# Trimmed sekvence (samo ECD)
seqs_trimmed = {
    "CLIC1":    fetched["CLIC1"]["ecd_seq"],
    "OSMR":     fetched["OSMR"]["ecd_seq"],
    "EGFRvIII": fetched["EGFR"]["viii_ecd_seq"],
}

# Tetramer CSV: CLIC1:OSMR:EGFRvIII:EGFRvIII
tetramer_seq = ":".join([seqs_trimmed["CLIC1"], seqs_trimmed["OSMR"],
                          seqs_trimmed["EGFRvIII"], seqs_trimmed["EGFRvIII"]])
tetramer_csv = os.path.join(jobname, f"{jobname}_tetramer.csv")
with open(tetramer_csv, "w") as f:
    f.write("id,sequence\n")
    f.write(f"{jobname},{tetramer_seq}\n")

# Pairwise CSV
names = list(seqs_trimmed.keys())
pairwise_csv = os.path.join(jobname, f"{jobname}_pairwise.csv")
with open(pairwise_csv, "w") as f:
    f.write("id,sequence\n")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            idname = f"{names[i]}_{names[j]}"
            seq = seqs_trimmed[names[i]] + ":" + seqs_trimmed[names[j]]
            f.write(f"{idname},{seq}\n")

total_res = sum(len(s) for s in tetramer_seq.split(":"))
print(f"\nTetramer CSV: {tetramer_csv}  (ukupno {total_res} AA)")
print(f"Pairwise CSV: {pairwise_csv}")
for k, v in seqs_trimmed.items():
    print(f"  {k}: {len(v)} aa")
if total_res > 2500:
    print("⚠️  WARNING: >2500 AA — rizik OOM. Koristi pairwise.")
else:
    print("✅ Ukupno AA OK za Colab GPU.")